## Opentrons Flex Script Guide

The following notebook provides a general guide for building scripts using the Opentrons API(application programming interface), as linked here:
[Opentrons API](https://docs.opentrons.com/v2/index.html)

Other tools for if you get stuck : 

#### Setting up a Script

For the Opentrons robot, protocols generally include similar first lines to set up the script:

In [ ]:
#import api 

from opentrons import types, protocol_api
from opentrons.protocol_api import ALL

#for any arithmetic expressions you may use in your script
from math import ceil, floor

You will then want to define the metadata. This will appear as the description of the protocol on the robot and inside the Opentrons Software.

In [ ]:
metadata = {
    'protocolName': 'SNAP: Sample Aliquoting',
    'author': 'Elizabeth Evin, evin@lanl.gov',
    'description': 'Aliquot samples from 96 well plate to more 96 well plates. Makes up 3 plates, mixes transcriptomics with buffer already added. Amended to remove sample prep steps.',
    'source': 'Opentrons'
}

The requirements section defines the robot, as well as the version of the API. Opentrons is constantly updating their API to include different modules and actions, and some modules, such as the Magnetic Block and the Plate Reader only appear in newer versions of the API. Make sure your API is up to date.

In [ ]:
requirements = {
    "robotType": "Flex",
    "apiLevel": "2.22"
}

#### Defining Parameters
The parameters are values that you want to be changeable in each run, ie you set the value on the robot before a run. Parameters can include number of samples, sample volumes, which wells to draw from in a reservoir, whether to change tips or not, honestly anything that can be defined by a float, integer, string or boolean value.

Below are examples of some common parameters:

In [ ]:
def add_parameters(parameters):
    parameters.add_float(
        variable_name="sample_volume",
        display_name="Metabolomics Sample Volume",
        description="What volume of sample to dispense into plate 1",
        default=200,
        minimum=5,
        maximum=1000
    )

The above parameter is first called by defining the add_parameters section of the script. The type of parameter(float) refers to this number as being a non discrete value, meaning that it can include decimals in the potential input options. The variable name will be the name that you refer to this number as in the script, and the display name is the volume the robot will display on its screen.

Below is an example of an integer parameter, which can only contain discrete values, such as the number of plates, or the number of samples to run.

In [ ]:
def add_parameters(parameters):
    parameters.add_int(
        variable_name='num_samples',
        display_name='Number of Samples',
        description='How many samples to aliquot on each plate',
        default=96,
        minimum=1,
        maximum=96
    )
    

The example below shows how to use a string parameter to indicate a well in a 12-well reservior. 

In [ ]:
def add_parameters(parameters):    
    parameters.add_str(
        variable_name="pooling_column",
        display_name="Pooling Destination Column",
        description="Column in reservoir to pool samples into",
        default="A1",
        choices=[
            {"display_name": "Column 1 (A1)", "value": "A1"},
            {"display_name": "Column 2 (A2)", "value": "A2"},
            {"display_name": "Column 3 (A3)", "value": "A3"},
            {"display_name": "Column 4 (A4)", "value": "A4"},
            {"display_name": "Column 5 (A5)", "value": "A5"},
            {"display_name": "Column 6 (A6)", "value": "A6"},
            {"display_name": "Column 7 (A7)", "value": "A7"},
            {"display_name": "Column 8 (A8)", "value": "A8"},
            {"display_name": "Column 9 (A9)", "value": "A9"},
            {"display_name": "Column 10 (A10)", "value": "A10"},
            {"display_name": "Column 11 (A11)", "value": "A11"},
            {"display_name": "Column 12 (A12)", "value": "A12"}
        ]
    )

Finally, the following example indicates how to use a boolean parameter (True/False) to indicate a dry run (where tips are returned to starting positions instead of discarded)

In [ ]:
def add_parameters(parameters): 
    parameters.add_bool(
        variable_name="dry_run",
        display_name="Dry Run",
        description="Off for real run, on for testing run to return tips",
        default=False
    )

## Writing the Run

Much of the work of the script is defined in the "run" function of the protocol. Below is an example of how to begin your run. Everything under the define run script should be nested below it, meaning that every line should be indented at least a tab in from the left by default. 

You must also name your run, and use that name to reference most run functions. In the below example, this run is named "protocol". To load in modules to this run, you must begin with the protocol name (protocol) and then finish the function call with the "load" command.

For the example below, making sure to load your waste chute is an essential part of the run, and invariable, so I typically add it first.

In [ ]:
def run(protocol: protocol_api.ProtocolContext):
    waste_chute = protocol.load_waste

### Calling Parameters

As the run starts, and the parameters are set, those values can be used similar to any other variable. However, to reference parameters as variables, it's best to read them into the script to make them easily accesible and to create a version of the parameter at the beginning that isn't changing. The example below demonstrates this well.

In [ ]:
def run(protocol: protocol_api.ProtocolContext):
    # Runtime parameters
    params = protocol.params
    sample_count = params.num_samples
    pooling_col = params.pooling_column
    dry_run = params.dry_run

First, you call the parameters as a whole by using the params function. Then you call individual parameters, defining them with the argument. Now, we have named objects that we can call and reference further along in our script.

### Defining Labware, Tips and Modules

Based on examples above, you already know how to load labware. Each labware piece must be loaded individually, and typically must be loaded with a location. Alternatively, labware can be loaded to the location "protocol_api.OFF_DECK", in case you want labware to begin the run off deck, and move later to the deck.
Below are some common labware that you may work with and load.

In [ ]:
def run(protocol: protocol_api.ProtocolContext):
    reservoir = protocol.load_labware("nest_1_reservoir_290ml", "B2",label="100% Ethanol") 
    sample_plate = protocol.load_labware("nest_96_wellplate_2ml_deep", "B3")
    reservoir2 = protocol.load_labware("nest_1_reservoir_290ml", protocol_api.OFF_DECK)


The labware piece is given a name "reservoir" which you can use to reference that piece of labware at that location. By calling the api name of the labware ("nest_1_reservoir_290mL") you have loaded in information about the labwre piece such as number of wells, well size, depth, height, width, and other important details. The robot now has a preset idea of how to perform funtions on the piece of labware. The "B2" location tells the robot where the labware is, according to the map below.

![labware map](https://docs.opentrons.com/v2/_images/initial-deck-map-flex.png)

It may be useful to print out this map or draw your own as you construct your own scripts. Keeping track of labware and not loading on top of other labware is essential to a functional script.

#### Tips and Pipette

Pipette tips are loaded in one of two ways. They can be loaded just as their labware name, similar to any other piece of labware (line 2 below). However, a 96 well head will require a tip adapter to pick up all 96 tips at once (line 3 & 4 below). This is performed by first loading the tip adapter, and then treating the adapter as a subset of the protocol and loading labware directly on to it. 
Additionally, using the 96 well head for a partial pickup will not require an adapter, which you will also need to plan for accordingly, especially when using a combination of pickup styles in a script.

In [ ]:
def run(protocol: protocol_api.ProtocolContext):
    tips = protocol.load_labware("opentrons_flex_96_tiprack_200ul", "B3" label="tip rack 1")
    tip_adapter1 = protocol.load_adapter("opentrons_flex_96_tiprack_adapter", "B2")
    tips2 = tip_adapter1.load_labware("opentrons_flex_96_tiprack_50ul", label="tip rack 2")

After you've loaded your tips, you can now load your pipette:

In [ ]:
def run(protocol: protocol_api.ProtocolContext):
    #for loading the 96 Channel
    left_pipette = protocol.load_instrument("flex_96channel_1000", tip_racks=[tips2])
    #for loading the 1 channel and the 8 channel (must load both)
    left_pipette = protocol.load_instrument("flex_1channel_1000", tip_racks=[tips])
    right_pipette = protocol.load_instrument("flex_8channel_1000", tip_racks=[tips])


#### Custom Labware

While much of the labware traditionally used in the Opentrons robots already exists in the [Labware Library](https://labware.opentrons.com/?_gl=1*6z52qh*_ga*MTM4NDA4NTUyMi4xNzUyNTk2MzE2*_ga_66HK7MC5D7*czE3NTc0Mzk4ODIkbzQ2JGcxJHQxNzU3NDQwMTEwJGo1OSRsMCRoMA..*_gcl_au*MTg5MTExMDk2Ni4xNzUyNTk2MzE2*_ga_GNSMNLW4RY*czE3NTc0Mzk4ODIkbzQ0JGcxJHQxNzU3NDQwMTEwJGo1OSRsMCRoNzY1OTI4ODY2#/) , custom labware is often used in protocols. This [Custom Labware Creator](https://labware.opentrons.com/?_gl=1*6z52qh*_ga*MTM4NDA4NTUyMi4xNzUyNTk2MzE2*_ga_66HK7MC5D7*czE3NTc0Mzk4ODIkbzQ2JGcxJHQxNzU3NDQwMTEwJGo1OSRsMCRoMA..*_gcl_au*MTg5MTExMDk2Ni4xNzUyNTk2MzE2*_ga_GNSMNLW4RY*czE3NTc0Mzk4ODIkbzQ0JGcxJHQxNzU3NDQwMTEwJGo1OSRsMCRoNzY1OTI4ODY2#/create) provides a resource to create a json file for a labware that does not exist in the current api. You can set all measurements, you can even create a labware file for a stacked labware piece, such as a filter plate on top of a 96 well plate, or a metal adapter with a 96 well plate.

This labware can then be downloaded, uploaded to the computer containing the opentrons software, and then used in scripts as a normal piece of labware. 

Some commonly used labware that can be found in this Github repository is used as an example below.

In [ ]:
def run(protocol: protocol_api.ProtocolContext):
    sample_plate = protocol.load_labware("eppendorf_96_wellplate_1000ul", "B3")
    working_plate = protocol.load_labware("eppendorf_96_tuberackshort_2000ul", "B3")
    final_plate =protocol.load_labware("quanrecovery_96_wellplate_700ul", "A2")
    

#### Modules

The interchangeable modules on the opentrons robot is one of the features that makes them so useful. Once your modules are set up in the software, you can use modules as labware locations as well. First, you must load them into the protocol to use them. Both examples below are correct for loading a module, defining the arguments with "module_name" and "location" can be useful to ensure clearness, but is not neccesary for simple functions.

In [ ]:
def run(protocol: protocol_api.ProtocolContext):
    hs_mod = protocol.load_module("heaterShakerModuleV1", "D1")
    temp_mod = protocol.load_module(module_name='temperature module gen2',location='C1')
    magnetic_block = protocol.load_module(module_name="magneticBlockV1", location="B1")

Next, you can load labware onto these modules, or leave them blank. Labware can later be moved to modules either manually or by robot. Similarly to tip adapters, labware is loaded directly to the module, and does not need a location.

In [ ]:
def run(protocol: protocol_api.ProtocolContext):
    sample_plate = hs_mod.load_labware("eppendorf_96_wellplate_1000ul")

Finally, you may need to perform certain commands to prepare the robot before a run when using specific modules. A common error is forgetting to close the labware latch for the heater-shaker module, which will throw an error for the run. To ensure it's always closed, the following line of code should be used:

In [ ]:
def run(protocol: protocol_api.ProtocolContext):
    hs_mod.close_labware_latch()

### Defining and loading liquids

While not a neccesary step, defining liquids can make it easier to use a script by displaying the need volumes and locations of initial reagents and samples. Below is an example of defining a liquid, and then loading that same liquid into a labware item. The unit for volume is uLs.

In [ ]:
def run(protocol: protocol_api.ProtocolContext):
    ethanol = protocol.define_liquid('Ethanol', display_color="#1d04ff")
    reservoir2["A1"].load_liquid(ethanol, volume = 10000)


The robot will now display the initial liquids and volumes on the screen for easy to follow loading.

### Run Commands

We have now entered the stage in the protocol development where you can actually code what you want the robot to *do*. Run commands can be simple or complex, iterate over entire plates, and incorporate movements, modules, and pauses. We will start with simplest run commands.

#### Transfer Commands

Transfer commands are likely the most useful run commands. The following transfer is specifically for a 96 channel pipette:

In [ ]:
left_pipette.transfer(500,reservoir2["A1"],sample_plate["A1"], new_tip="never")

The structure of this line of code indicates (volume,starting location,ending location, new_tip).
In this case, the "A1" refers to the well in the plate or reservoir, and in this case, an A1 transfer would refer to the full plate since we are using a 96 channel head.